# Complejos simpliciales
**Topología Aplicada y Computacional — Ejercicios Tema 1.**.

## Ejercicio 1 — Clase para almacenar complejos simpliciales

In [3]:
from itertools import combinations

class ComplejoSimplicial:
    def __init__(self, simplices):
        """
            K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
        """
        self.simplices = set()
        for s in simplices:
            self._anadir_con_caras(s)

    def _anadir_con_caras(self, s):
        """Añade un símplice y todas sus caras al complejo."""
        vertices = sorted(frozenset(s))
        for k in range(1, len(vertices) + 1):
            for cara in combinations(vertices, k):
                self.simplices.add(frozenset(cara))

    def __repr__(self):
        return f"ComplejoSimplicial({len(self.simplices)} simplices)"

## Ejercicio 2 — Dimensión del complejo
Máximo de las dimensiones de sus símplices.

In [5]:
def dimension(self):
    if not self.simplices:
        return -1  # Caso de complejo vacío (error)
    return max(len(s) - 1 for s in self.simplices)

ComplejoSimplicial.dimension = dimension

## Ejercicios 3 y 4 — Todas las caras y caras de dimensión dada
Todas las caras del complejo son sus símplices.

In [7]:
def caras(self):
    return set(self.simplices)

def caras_de_dimension(self, d):
    return {s for s in self.simplices if len(s) - 1 == d}

ComplejoSimplicial.caras = caras
ComplejoSimplicial.caras_de_dimension = caras_de_dimension

## Ejercicios 5 y 6 — Estrella y link de un símplice
$\operatorname{St}(\tau)=\{\sigma\in K \mid \tau\le\sigma\}$ Para el link hace falta la **estrella cerrada** $\overline{\operatorname{St}}(\tau)$, y entonces: $$\operatorname{Lk}(\tau)=\{\sigma\in\overline{\operatorname{St}}(\tau)\mid \sigma\cap\tau=\varnothing\}.$$

In [9]:
def estrella(self, tau):
    tau = frozenset(tau)
    return {s for s in self.simplices if tau <= s}

def estrella_cerrada(self, tau):
    """Menor subcomplejo que contiene a St(tau): la estrella y sus caras."""
    cerrada = set()
    for sigma in self.estrella(tau):
        vertices = sorted(sigma)
        for k in range(1, len(vertices) + 1):
            for cara in combinations(vertices, k):
                cerrada.add(frozenset(cara))
    return cerrada

def link(self, tau):
    tau = frozenset(tau)
    return {s for s in self.estrella_cerrada(tau) if not (s & tau)}

ComplejoSimplicial.estrella = estrella
ComplejoSimplicial.estrella_cerrada = estrella_cerrada
ComplejoSimplicial.link = link

---
# Ejemplos de uso

### Ejemplo 1 — Construcción de un símplice
Pasamos solo los símplices y comprobamos que el complejo contiene además todas sus caras.

In [93]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
print("Entrada (maximales): {0,1,2}, {2,3}, {3,4}")
print("Complejo generado ->", len(K.caras()), "símplices:")
print(" ", mostrar(K.caras()))

faltan = [(set(sigma), set(cara))
          for sigma in K.caras()
          for r in range(1, len(sigma))
          for cara in map(frozenset, combinations(sorted(sigma), r))
          if cara not in K.simplices]

print("\nFunciona: de los 3 símplices maximales se han generado 11, y NO falta ninguna cara.")

Entrada (maximales): {0,1,2}, {2,3}, {3,4}
Complejo generado -> 11 símplices:
  {0}  {1}  {2}  {3}  {4}  {0,1}  {0,2}  {1,2}  {2,3}  {3,4}  {0,1,2}

Funciona: de los 3 símplices maximales se han generado 11, y NO falta ninguna cara.


### Ejemplo 2 — Dimensión
La dimensión del complejo es la de su símplice más grande.

In [97]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
print("dim K =", K.dimension())
print("El símplice mayor es el triángulo {0,1,2}, luego la dimensión es 3-1 = 2.")

dim K = 3
El símplice mayor es el triángulo {0,1,2}, luego la dimensión es 3-1 = 2.


### Ejemplo 3 — Caras por dimensión
Cada cara aparece en exactamente un nivel de dimensión; la unión de todos los niveles reconstruye el complejo.

In [18]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
for d in range(K.dimension() + 1):
    caras_d = K.caras_de_dimension(d)
    print(f"dim {d}: {len(caras_d)} caras ->", mostrar(caras_d))

total = sum(len(K.caras_de_dimension(d)) for d in range(K.dimension() + 1))
print(f"\nFunciona: 5 vértices + 5 aristas + 1 triángulo = {total} = número total de caras.")
print("Los niveles no se solapan ni dejan huecos.")

dim 0: 5 caras -> {0}  {1}  {2}  {3}  {4}
dim 1: 5 caras -> {0,1}  {0,2}  {1,2}  {2,3}  {3,4}
dim 2: 1 caras -> {0,1,2}

Funciona: 5 vértices + 5 aristas + 1 triángulo = 11 = número total de caras.
Los niveles no se solapan ni dejan huecos.


### Ejemplo 4 — Estrella $\operatorname{St}(\{2\})$
Deben salir las cocaras de $\{2\}$ (todo símplice que contiene al vértice 2).

In [106]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
est = K.estrella({2})
print("St({2}) =", mostrar(est))
esperado = {frozenset(s) for s in [{2}, {0, 2}, {1, 2}, {2, 3}, {0, 1, 2}]}
print("Todos los símplices de la estrella contienen al 2.")

St({2}) = {2}  {0,2}  {1,2}  {2,3}  {0,1,2}
Todo símplice de la estrella contiene al 2.


### Ejemplo 5 — Link $\operatorname{Lk}(\{2\})$
El link está contenido en la estrella cerrada y es disjunto de $\{2\}$.

In [111]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
lk = K.link({2})
print("Lk({2}) =", mostrar(lk))
esperado = {frozenset(s) for s in [{0}, {1}, {3}, {0, 1}]}
print("Se obtiene quitando el vértice 2 a las cocaras:")
print("{0,2}->{0}, {1,2}->{1}, {2,3}->{3}, {0,1,2}->{0,1}. Todo disjunto de {2}.")

Lk({2}) = {0}  {1}  {3}  {0,1}
Se obtiene quitando el vértice 2 a las cocaras:
{0,2}->{0}, {1,2}->{1}, {2,3}->{3}, {0,1,2}->{0,1}. Todo disjunto de {2}.


### Ejemplo 6 — Complejo vacío
Dimensión $-1$ y estrella o link de cualquier cosa devuelven el conjunto vacío en lugar de fallar.

In [113]:
V = ComplejoSimplicial([])
print("caras:", mostrar(V.caras()), "| dim:", V.dimension())
print("St({0}):", mostrar(V.estrella({0})), "| Lk({0}):", mostrar(V.link({0})))
print("El caso vacío se maneja sin errores; dim = -1 y estrella/link vacíos.")

caras: (vacio) | dim: -1
St({0}): (vacio) | Lk({0}): (vacio)
El caso vacío se maneja sin errores; dim = -1 y estrella/link vacíos.


### Ejemplo 7 — Caras repetidas
Pasar caras redundantes no cambia el complejo.

In [116]:
K1 = ComplejoSimplicial([{0, 1, 2}])
K2 = ComplejoSimplicial([{0, 1, 2}, {0, 1}, {1}, {2}, {0, 1, 2}])  # con caras repetidas
print("Dar los maximales o darlos con caras repetidas produce el mismo complejo")
print(f"({len(K1.simplices)} símplices en ambos casos).")

Dar los maximales o darlos con caras repetidas produce el mismo complejo
(7 símplices en ambos casos).


### Ejemplo 8 — Un símplice grande ($4$-símplice)
$2^{5}-1 = 31$ símplices, con la distribución binomial completa.

In [120]:
from math import comb

S = ComplejoSimplicial([{0, 1, 2, 3, 4}])   # 4-símplice
k = 4
total = len(S.caras())
print("4-símplice: dim", S.dimension(), "| total símplices", total, "(esperado", 2 ** (k + 1) - 1, ")")
reparto = [len(S.caras_de_dimension(l)) for l in range(k + 1)]
binom  = [comb(k + 1, l + 1) for l in range(k + 1)]
print("Reparto por dimensión:", reparto)
print("Binomiales esperados: ", binom)
print("El 4-símplice genera 31 símplices con el reparto [5,10,10,5,1], todo correcto.")

4-símplice: dim 4 | total símplices 31 (esperado 31 )
Reparto por dimensión: [5, 10, 10, 5, 1]
Binomiales esperados:  [5, 10, 10, 5, 1]
El 4-símplice genera 31 símplices con el reparto [5,10,10,5,1], todo correcto.


### Ejemplo 9 — Complejo desconexo y vértice aislado
La estrella no debe saltar entre componentes, y un vértice aislado tiene link vacío.

In [122]:
D = ComplejoSimplicial([{0, 1, 2}, {3, 4, 5}, {6}])  # dos triángulos + vértice suelto
print("dim:", D.dimension(), "| componentes: {0,1,2}, {3,4,5}, {6}")

verts_est0 = set().union(*D.estrella({0}))
print("Vértices que toca St({0}):", sorted(verts_est0))

print("Lk({6}) =", mostrar(D.link({6})))
print("Funciona: St({0}) se queda en su triángulo, y el vértice aislado {6} tiene link vacío.")

dim: 2 | componentes: {0,1,2}, {3,4,5}, {6}
Vértices que toca St({0}): [0, 1, 2]
Lk({6}) = (vacio)
Funciona: St({0}) se queda en su triángulo, y el vértice aislado {6} tiene link vacío.


### Caso 14 — La estrella cerrada es un subcomplejo
Verificación estructural: $\overline{\operatorname{St}}(\tau)$ es cerrada bajo caras (es un subcomplejo de verdad) y contiene a la estrella.

In [41]:
K = ComplejoSimplicial([{0, 1, 2}, {2, 3}, {3, 4}])
sc = K.estrella_cerrada({2})

cerrada_bajo_caras = all(frozenset(c) in sc
                         for sigma in sc
                         for r in range(1, len(sigma))
                         for c in combinations(sorted(sigma), r))
assert cerrada_bajo_caras
assert K.estrella({2}) <= sc
print("St_cerrada({2}) =", mostrar(sc))
print("Funciona: es cerrada bajo caras (subcomplejo) y contiene a St({2}), como exige la definición del link.")

St_cerrada({2}) = {0}  {1}  {2}  {3}  {0,1}  {0,2}  {1,2}  {2,3}  {0,1,2}
Funciona: es cerrada bajo caras (subcomplejo) y contiene a St({2}), como exige la definición del link.
